# Week 2 — Complete Preprocessing and ML Modelling

Build a leakage-safe pipeline for one selected CSV, compare baseline models, and persist the complete fitted pipeline. Default: `heart_disease.csv` → `Heart Disease Status`. Change the constants for another task.


In [1]:
from pathlib import Path
import re, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, balanced_accuracy_score
import joblib
ROOT = Path.cwd()
if not (ROOT/'heart_disease.csv').exists() and (ROOT.parent/'heart_disease.csv').exists(): ROOT = ROOT.parent
DATASET_NAME, TARGET_COLUMN, RANDOM_STATE = 'heart_disease', 'Heart Disease Status', 42
df = pd.read_csv(ROOT/f'{DATASET_NAME}.csv'); df.columns = [re.sub(r'\s+', ' ', str(c)).strip() for c in df.columns]; df = df.drop_duplicates()
if TARGET_COLUMN not in df: raise ValueError(f'Missing target {TARGET_COLUMN!r}; available columns: {list(df.columns)}')
X, y = df.drop(columns=[TARGET_COLUMN]), df[TARGET_COLUMN].astype(str).str.strip()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, stratify=y, random_state=RANDOM_STATE)
num = X_train.select_dtypes(include=np.number).columns.tolist(); cat = [c for c in X_train.columns if c not in num]
pre = ColumnTransformer([('num', Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]), num), ('cat', Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]), cat)])
print('Target distribution:'); display(y.value_counts().to_frame('count')); print(f'Numeric={len(num)}, categorical/text={len(cat)}')


Target distribution:


,count
Heart Disease Status,
No,8000
Yes,2000


Numeric=9, categorical/text=11


In [2]:
models = {'logistic_regression':LogisticRegression(max_iter=1500, class_weight='balanced', random_state=RANDOM_STATE), 'random_forest':RandomForestClassifier(n_estimators=250, class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE)}
results, fitted = [], {}
for name, estimator in models.items():
    pipe = Pipeline([('preprocessor',pre),('model',estimator)]); pipe.fit(X_train,y_train); pred = pipe.predict(X_test)
    results.append({'model':name, 'balanced_accuracy':balanced_accuracy_score(y_test,pred)}); fitted[name] = pipe
    print(f'\n{name}\n', classification_report(y_test,pred,zero_division=0))
results_df = pd.DataFrame(results).sort_values('balanced_accuracy', ascending=False); display(results_df)
best_name = results_df.iloc[0].model; best_pipeline = fitted[best_name]
model_dir = ROOT/'artifacts/models'; model_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(best_pipeline, model_dir/f'{DATASET_NAME}_{best_name}.joblib'); results_df.to_csv(ROOT/'artifacts/week2_model_results.csv', index=False)
print('Saved', model_dir/f'{DATASET_NAME}_{best_name}.joblib')



logistic_regression
               precision    recall  f1-score   support

          No       0.79      0.51      0.62      1600
         Yes       0.19      0.47      0.27       400

    accuracy                           0.50      2000
   macro avg       0.49      0.49      0.45      2000
weighted avg       0.67      0.50      0.55      2000




random_forest
               precision    recall  f1-score   support

          No       0.80      1.00      0.89      1600
         Yes       0.00      0.00      0.00       400

    accuracy                           0.80      2000
   macro avg       0.40      0.50      0.44      2000
weighted avg       0.64      0.80      0.71      2000



,model,balanced_accuracy
1,random_forest,0.498750
0,logistic_regression,0.487187


Saved /home/ubuntu/projecttest/artifacts/models/heart_disease_random_forest.joblib


In [3]:
# Optional image branch: verify downloaded images are a separate input stream.
try:
    from PIL import Image
    image_dirs = [p for p in [ROOT/'data/images',ROOT/'images',ROOT/'Lung X-Ray Image',ROOT/'skin-disease-images'] if p.exists()]
    sample = next((p for d in image_dirs for p in d.rglob('*') if p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.webp'}), None)
    if sample:
        with Image.open(sample) as im: print('Image:',sample,'size:',im.size,'mode:',im.mode); display(im.convert('RGB').resize((224,224)))
except ImportError: print('Install Pillow to inspect images.')


## Exit criteria

Confirm the target is appropriate, review imbalance, preserve preprocessing and estimator together, and keep image preprocessing separate from the tabular/text pipeline. A production image model needs its own validated CNN or transfer-learning branch.
